In [76]:
import re
import gspread
import pandas as pd
import numpy as np

from google.oauth2.service_account import Credentials

## FAQ Section

In [6]:
SERVICE_ACCOUNT_FILE = 'key/Credentials.json'
SCOPES = ['https://www.googleapis.com/auth/spreadsheets',
          'https://www.googleapis.com/auth/drive']
creds = Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES
)
client = gspread.authorize(creds)

sheet = client.open('NEW Akulaku - AI Kula Scene Classification (2025)').worksheet('NEW 12.14')
data = sheet.get_all_records()

df = pd.DataFrame(data[0:], columns=data[0])

In [7]:
# cleaning the columns
df_clean = df.copy()

def remove_mandarin(text):
    return re.sub('r[\u4e00-\u9fff]+', '', str(text))

df_clean.columns = (
    df_clean.columns
    .map(remove_mandarin)
    .str.replace('\n', '')
    .str.replace(' ','_')
    .str.replace(r'[^A-Za-z0-9_()-]', '', regex=True)
    .str.replace('-','_')
    .str.lower()
    .str.strip('_')
)

df_clean = df_clean.apply(lambda x: x.lower() if isinstance(x, str) else x)

df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 923 entries, 0 to 922
Data columns (total 57 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0                                923 non-null    object
 1   no                           923 non-null    int64 
 2   create_time                  923 non-null    object
 3   classification_business      923 non-null    object
 4   faq_status                   923 non-null    object
 5   old_code                     923 non-null    object
 6   new_code                     923 non-null    object
 7   kategori_1_(cn)              923 non-null    object
 8   kategori_1_(en)              923 non-null    object
 9   kategori_1_(id)              923 non-null    object
 10  kategori_2_(cn)              923 non-null    object
 11  kategori_2_(en)              923 non-null    object
 12  kategori_2_(id)              923 non-null    object
 13  kategori_3_(cn)              923 no

In [8]:
# The raw data
raw = [
    '''
1334-1335-1134
1662-1663-1664-996
289-1527-1528-922
289-1562-1563-943
289-1667-1668-998
289-1770-1771-1188
289-1821-1822-1217
289-292-446-173
289-303-460-349
289-308-465-411
289-309-466-412
289-317-474-477
289-326-483-503
289-327-484-505
289-328-485-506
289-332-489-510
289-338-495-516
289-364-521-570
289-369-526-713
289-369-526-777
289-374-531-610
289-380-537-622
289-382-539-624
289-397-554-652
289-398-555-656
289-409-566-712
289-410-567-715
289-414-571-735
289-415-572-736
289-416-573-737
289-419-576-751
289-423-580-772
289-424-581-778
289-425-582-816
289-433-590-834
289-435-592-846
289-437-594-849
695-701-708-740
710-711-717-440
710-713-719-720
710-714-720-727
710-715-721-748
738-1608-1609-967
738-739-758-121
738-755-774-837
74-169-276-815
794-797-811-214
794-805-819-752
794-807-821-810
872-1721-1163
872-1736-1737-1171
872-890-985-243
1334-1069
1334-1335-1049
1334-1335-1051
1334-1335-1059
1334-1335-1062
289-1714-1715-1160
872-1787-1788-1200
1-18-62-422
1134-1156-1179-807
1334-1335-1065
1334-1335-1072
1334-1335-1107
1334-1335-1116
1334-1335-1121
1334-1335-1123
1344-1356-1418-260
1344-1365-1427-278
1344-1398-1460-653
1344-1400-1462-677
1344-1402-1464-680
872-949-1044-769
872-951-1046-774
1344-1349-1411-233
872-900-995-372
872-915-1010-494
872-922-1017-527
289-398-555-656
872-890-985-243
289-326-483-503
872-1721-1163
872-900-995-372
872-922-1017-527
289-317-474-477
872-1736-1737-1171
872-915-1010-494
738-755-774-837
289-424-581-778
289-292-446-173
289-369-526-777
289-1562-1563-943
289-374-531-610
1344-1365-1427-278
1344-1349-1411-233
1344-1356-1418-260
1134-1156-1179-807
1-18-62-422
'''
]

code_list = pd.DataFrame(raw[0].split('\n'), columns=['code_list'])

code_list = code_list.replace('','-').drop_duplicates(keep=False)
code_list

,code_list
1,1334-1335-1134
2,1662-1663-1664-996
3,289-1527-1528-922
5,289-1667-1668-998
6,289-1770-1771-1188
7,289-1821-1822-1217
9,289-303-460-349
10,289-308-465-411
11,289-309-466-412
14,289-327-484-505


In [11]:
'''
raw_2 = [
    '''
1063-1064-1083-159
1063-1072-1112-310
1063-1072-1114-312
1063-1072-1115-313
1-9-43-229
1-9-44-860
1-9-45-861
1063-1069-1101-205
1063-1069-1102-206
1063-1069-1103-207
289-416-573-737
620-636-673-385
1063-1067-1094-200
1063-1067-1095-871
1063-1067-1096-878
'''
]

raw_2_df = pd.DataFrame(raw_2[0].split('\n'), columns=['code_list'])

raw_2_series = pd.Series(raw_2[0].split('\n')).reset_index(drop=True)
raw_2_series = raw_2_series.replace('','-').drop_duplicates(keep=False)

# comparing raw and raw 2
raw_series = raw.iloc[:, 0].reset_index(drop=True)
raw_2_series = raw_2_series.reset_index(drop=True)

checking = pd.DataFrame({
    'raw': raw_series,
    'raw_2': raw_2_series,
})

checking['is_duplicated'] = (checking['raw'].str.strip() == checking['raw_2'].str.strip())

checking
'''

"\n]\n\nraw_2_df = pd.DataFrame(raw_2[0].split('\n'), columns=['code_list'])\n\nraw_2_series = pd.Series(raw_2[0].split('\n')).reset_index(drop=True)\nraw_2_series = raw_2_series.replace('','-').drop_duplicates(keep=False)\n\n# comparing raw and raw 2\nraw_series = raw.iloc[:, 0].reset_index(drop=True)\nraw_2_series = raw_2_series.reset_index(drop=True)\n\nchecking = pd.DataFrame({\n    'raw': raw_series,\n    'raw_2': raw_2_series,\n})\n\nchecking['is_duplicated'] = (checking['raw'].str.strip() == checking['raw_2'].str.strip())\n\nchecking\n"

In [12]:
raw_3 = [
    '''
1334-1335-1134
1662-1663-1664-996
289-1527-1528-922
289-1562-1563-943
289-1667-1668-998
289-1770-1771-1188
289-1821-1822-1217
289-292-446-173
289-303-460-349
289-308-465-411
289-309-466-412
289-317-474-477
289-326-483-503
289-327-484-505
289-328-485-506
289-332-489-510
289-338-495-516
289-364-521-570
289-369-526-713
289-369-526-777
289-374-531-610
289-380-537-622
289-382-539-624
289-397-554-652
289-398-555-656
289-409-566-712
289-410-567-715
289-414-571-735
289-415-572-736
289-416-573-737
289-419-576-751
289-423-580-772
289-424-581-778
289-425-582-816
289-433-590-834
289-435-592-846
289-437-594-849
695-701-708-740
710-711-717-440
710-713-719-720
710-714-720-727
710-715-721-748
738-1608-1609-967
738-739-758-121
738-755-774-837
74-169-276-815
794-797-811-214
794-805-819-752
794-807-821-810
872-1721-1163
872-1736-1737-1171
872-890-985-243
1334-1069
1334-1335-1049
1334-1335-1051
1334-1335-1059
1334-1335-1062
289-1714-1715-1160
872-1787-1788-1200
1-18-62-422
1134-1156-1179-807
1334-1335-1065
1334-1335-1072
1334-1335-1107
1334-1335-1116
1334-1335-1121
1334-1335-1123
1344-1356-1418-260
1344-1365-1427-278
1344-1398-1460-653
1344-1400-1462-677
1344-1402-1464-680
872-949-1044-769
872-951-1046-774
1344-1349-1411-233
872-900-995-372
872-915-1010-494
872-922-1017-527
289-398-555-656
872-890-985-243
289-326-483-503
872-1721-1163
872-900-995-372
872-922-1017-527
289-317-474-477
872-1736-1737-1171
872-915-1010-494
738-755-774-837
289-424-581-778
289-292-446-173
289-369-526-777
289-1562-1563-943
289-374-531-610
1-18-62-422
1063-1064-1083-159
1063-1072-1112-310
1063-1072-1114-312
1063-1072-1115-313
1-9-43-229
1-9-44-860
1-9-45-861
1063-1069-1101-205
1063-1069-1102-206
1063-1069-1103-207
289-416-573-737
620-636-673-385
1063-1067-1094-200
1063-1067-1095-871
1063-1067-1096-878
'''
]

raw_3 = pd.DataFrame(raw_3[0].split('\n'), columns=['code_list'])
raw_3[raw_3.duplicated()]

,code_list
79,289-398-555-656
80,872-890-985-243
81,289-326-483-503
82,872-1721-1163
83,872-900-995-372
84,872-922-1017-527
85,289-317-474-477
86,872-1736-1737-1171
87,872-915-1010-494
88,738-755-774-837


In [13]:
# # Imputating the holy faq
# kategori = df_clean.iloc[:,[6,9,12,15,18,21,47]]

# # Comparing FAQ
# result = raw_2_df.merge(kategori, left_on='code_list', right_on='new_code', how='left')
# result.to_csv('temp/faq.csv')

## Chit Chat Section

In [ ]:
# Pull
# sheet = client.open('NEW Akulaku - AI Kula Scene Classification (2025)').worksheet('Chit-Chat Added')
# data = sheet.get_all_records()

# df_chit_chat = pd.DataFrame(data[0:], columns=data[0])
# df_chit_chat.to_csv('temp/faq_chit_chat.csv')
df_chit_chat = pd.read_csv('temp/faq_chit_chat.csv')

# clean the column
df_chit_chat.columns = (
    df_chit_chat.columns
    .map(remove_mandarin)
    .str.replace('\n', '')
    .str.replace(' ','_')
    .str.replace(r'[^A-Za-z0-9_()-]', '', regex=True)
    .str.replace('-','_')
    .str.lower()
    .str.strip('_')
)

# Imputation
df_chit_chat = df_chit_chat.iloc[:,[3,5,6,7,10,13,14]]

# removing extra spaces
df_chit_chat = df_chit_chat.apply(
    lambda x: x.astype(str).str.strip() if x.dtype == 'object' else x
)

df_chit_chat

,code,penjelasan_skenario_2,penjelasan_skenario_3,id_example_(id),answer_(id),testing_online_chit_chat111024,answer_false
0,1334-1335-1048,Chitchat,Basa basi,Assalamualaikum,Halo Kak. Terima kasih sudah chat Kula,True,nan
1,1334-1335-1049,Chitchat,Basa basi,Min,"Iya Kak, ada yang bisa Kula bantu?",False,"Maaf, kula tidak mengerti apa yang Anda maksud..."
2,1334-1335-1050,Chitchat,Basa basi,Kakak,Halo Kak. Ada yang bisa Kula bantu?,True,nan
3,1334-1335-1051,Chitchat,Basa basi,Mau tanya,Hai Kak. Terima kasih sudah chat Kula. Ada yan...,True,nan
4,1334-1335-1052,Chitchat,Basa basi,Maaf Kak,"Iya Kak, ada yang bisa Kula bantu?",True,nan
...,...,...,...,...,...,...,...
85,1334-1335-1104,Chitchat,Basa basi,pe,"Maaf Kak, bisa diinformasikan kembali kendalanya?",False,"Maaf, kula tidak mengerti apa yang Anda maksud..."
86,1334-1335-1105,Chitchat,Basa basi,pylter,"Maaf Kak, bisa diinformasikan kembali kendalanya?",False,"Maaf, kula tidak mengerti apa yang Anda maksud..."
87,1334-1335-1106,Chitchat,Basa basi,jalo,"Maaf Kak, bisa diinformasikan kembali kendalanya?",False,"Maaf, kula tidak mengerti apa yang Anda maksud..."
88,1334-1335-1127,Chitchat,Basa basi,lakukan yg terbaik,Kula siap bantu kamu. Apa yang bisa Kula bantu?,True,nan


In [ ]:
impute_chit_chat = result[result['new_code'].isna()]
impute_chit_chat

,code_list,new_code,kategori_1_(id),kategori_2_(id),kategori_3_(id),idbackground_detail__id,idexample,new___answer
2,1334-1069,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1334-1335-1049,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1334-1335-1051,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1334-1335-1059,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1334-1335-1062,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1334-1335-1065,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1334-1335-1072,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1334-1335-1107,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,1334-1335-1116,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,1334-1335-1121,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Comparing chit-chat
result_chit_chat = impute_chit_chat.merge(df_chit_chat, left_on='code_list', right_on='code', how='left')

# giving out
result_chit_chat.to_csv('temp/merge_faq_chit_chat.csv')

result_chit_chat

,code_list,new_code,kategori_1_(id),kategori_2_(id),kategori_3_(id),idbackground_detail__id,idexample,new___answer,code,penjelasan_skenario_2,penjelasan_skenario_3,id_example_(id),answer_(id),testing_online_chit_chat111024,answer_false
0,1334-1069,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1334-1069,Chitchat,Basa basi,Kak,Halo Kak. Ada yang bisa Kula bantu?,True,nan
1,1334-1335-1049,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1334-1335-1049,Chitchat,Basa basi,Min,"Iya Kak, ada yang bisa Kula bantu?",False,"Maaf, kula tidak mengerti apa yang Anda maksud..."
2,1334-1335-1051,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1334-1335-1051,Chitchat,Basa basi,Mau tanya,Hai Kak. Terima kasih sudah chat Kula. Ada yan...,True,nan
3,1334-1335-1059,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1334-1335-1059,Chitchat,Basa basi,Sudah,Baik Kak. Terima kasih\n\nBaik Kak. Ada yang b...,False,https://prnt.sc/4TEs4LDMKrJV \nHai Kak! Mohon ...
4,1334-1335-1062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1334-1335-1062,Chitchat,Basa basi,tapi kok sdah ada tagihan,Mohon maaf ya Kak sudah buat Kakak tidak nyama...,nan,nan
5,1334-1335-1065,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1334-1335-1065,Chitchat,Basa basi,siang,Halo Kak. Ada yang bisa Kula bantu?,True,nan
6,1334-1335-1072,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1334-1335-1072,Chitchat,Basa basi,hallo,Halo Kak. Terima kasih sudah chat Kula,True,nan
7,1334-1335-1107,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1334-1335-1107,Chitchat,Basa basi,selamat malam,"Makasih sudah chat Kula, apa masih ada yang bi...",True,nan
8,1334-1335-1116,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1334-1335-1116,Chitchat,Basa basi,pagi,Halo Kak. Ada yang bisa Kula bantu?,True,nan
9,1334-1335-1121,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1334-1335-1121,Chitchat,Basa basi,minta tolong,"Iya Kak, ada yang bisa Kula bantu?",True,nan


## Reverse Pipeline

In [27]:
# Pull top complaint data (ASI)
df_asi = df_clean.copy()

df_asi = (
    df_asi[df_asi['classification_business'] == 'ASI']
    .iloc[:,[2,3,6,9,12,15,18,21,44,47]]
)

# date data type
df_asi['create_time'] = pd.to_datetime(df_asi['create_time'])

# sorting
df_asi = df_asi.sort_values(by='create_time', axis=0, ascending=False)

# fillna idanswer
df_asi['new___answer'] = (
    df_asi['new___answer']
    .replace(r'^\s*$', pd.NA, regex=True)
    .fillna(df_asi['idanswer'])
)

#export
df_asi.to_csv('temp/latest_faq_asi.csv')

df_asi

,create_time,classification_business,new_code,kategori_1_(id),kategori_2_(id),kategori_3_(id),idbackground_detail__id,idexample,idanswer,new___answer
921,2026-02-10,ASI,289-1860-1861-1236,Angsuran dan Pinjaman,Cara bayar tagihan pinjaman via GOPAY,Pinjaman,Cara bayar tagihan pinjaman via GOPAY,Bagaimana cara melakukan pembayaran tagihan pi...,"""Berikut kami informasikan langkah-langkah pem...","""Berikut kami informasikan langkah-langkah pem..."
917,2025-10-01,ASI,289-1852-1853-1232,Angsuran dan Pinjaman,Keringanan/diskon denda Pinjaman,Pinjaman,Keringanan/diskon denda Pinjaman,Apakah bisa meminta keringanan tagihan dan den...,,Kula bantu informasikan untuk melakukan proses...
911,2025-08-21,ASI,1344-1840-1841-1226,Virtual,Tidak bisa top-up E-Wallet,Voucher,Tidak bisa top-up E-Wallet,Kenapa saya tidak bisa top up e wallet di apli...,,"Mohon maaf atas ketidaknyamanannya ya, Kak. Ku..."
906,2025-07-29,ASI,289-1830-1831-1221\t,Angsuran dan Pinjaman,Perpanjang tenor Pinjaman,Pinjaman,Perpanjang jatuh tempo Pinjaman,Apakah akun pinjam saya dapat di perpanjang ja...,,"Untuk saat ini, tenor pinjaman hanya tersedia ..."
903,2025-07-29,ASI,289-1824-1825-1218,Angsuran dan Pinjaman,Tidak bisa bayar tagihan Pinjaman via OVO,Pinjaman,Tidak bisa bayar tagihan Pinjaman via OVO,Kenapa pembayaran tagihan pinjaman OVO tidak ada?,,"Halo Kak, mohon maaf atas ketidaknyamanannya.\..."
...,...,...,...,...,...,...,...,...,...,...
894,NaT,ASI,1181-1793-1794-1203\t,Promo/Undian/Event,[Lucky Draw] - Hadiah Voucher belanja tidak bi...,Promo/Undian/Event,[Lucky Draw] - Hadiah Voucher belanja tidak bi...,Saya sudah memenangkan hadiah voucher belanja ...,,Mohon maaf untuk jenis permasalahan ini memerl...
895,NaT,ASI,1181-1795-1796-1204,Promo/Undian/Event,[Lucky Draw] - Tidak mendapatkan hadiah tunai,Promo/Undian/Event,[Lucky Draw] - Tidak mendapatkan hadiah tunai,Saya sudah memainkan Lucky Draw / Lucky Spin b...,,Mohon maaf untuk jenis permasalahan ini memerl...
897,NaT,ASI,289-1811-1812-1212\t,Angsuran dan Pinjaman,Bayar tagihan Pinjaman pakai rekening orang lain,Pinjaman,Bayar tagihan Pinjaman pakai rekening orang lain,min kalo mau bayar tagihan Pinjaman nama pengi...,,"Tentu, pembayaran tagihan bisa dilakukan mengg..."
898,NaT,ASI,1181-1813-1814-1213\t,Promo/Undian/Event,[Lucky Draw] - Kouta habis,Promo/Undian/Event,[Lucky Draw] - Kouta habis,Kenapa hadiah Lucky Draw diganti jadi poin saa...,,Mohon maaf untuk jenis permasalahan ini memerl...


In [17]:
# Pull top complaint data (ASI)
df_afi = df_clean.copy()

df_afi = (
    df_afi[df_afi['classification_business'] == 'AFI']
    .iloc[:,[2,3,6,9,12,15,18,21,44,47]]
)

# date data type
df_afi['create_time'] = pd.to_datetime(df_afi['create_time'])

# sorting
df_afi = df_afi.sort_values(by='create_time', axis=0, ascending=False)

#export
df_afi.to_csv('temp/latest_faq_asi.csv')

df_afi

,create_time,classification_business,new_code,kategori_1_(id),kategori_2_(id),kategori_3_(id),idbackground_detail__id,idexample,idanswer,new___answer
920,2026-02-10,AFI,872-1858-1859-1235,Payment,Bayar tagihan Paylater via OVO,Installment/Payment Gateway,Bayar tagihan Paylater Via OVO,Bagaimana Cara bayar tagihan Akulaku Paylater ...,"Kula bantu informasikan ya, Kak. Berikut adala...",
919,2026-02-10,AFI,872-1856-1857-1234,Payment,Bayar tagihan Paylater Via GOPAY,Installment/Payment Gateway,Bayar tagihan Paylater Via GOPAY,Bagaimana Cara bayar tagihan Akulaku Paylater ...,Berikut Kula informasikan langkah-langkah pemb...,
916,2025-09-17,AFI,289-1850-1851-1231,Angsuran dan Pinjaman,Evaluasi Skor Kredit,Pengajuan limit/kredit poin,Evaluasi Skor Kredit,Kenapa kredit skor saya kembali ke awal.\npada...,,"Kula informasikan, terkait skor kredit perlu K..."
915,2025-09-17,AFI,74-1848-1849-1230\t,Akulaku Paylater,Lazada Paylater - Tidak bisa pembayaran karena...,Akulaku Paylater,Lazada Paylater - Tidak bisa pembayaran karena...,Mau ngelunasi Lazada Paylater gimana sedangkan...,,"Kula bantu informasikan, terkait pertanyaan Ka..."
914,2025-09-17,AFI,695-1846-1847-1229\t,Collection,Cara Restrukturisasi tagihan,Collection,Cara Restrukturisasi tagihan,Apakah saya bisa mengajukan reksturktur angsuran?,,"Mohon maaf Kak, Kula informasikan saat ini pro..."
...,...,...,...,...,...,...,...,...,...,...
247,2022-12-01,AFI,872-897-992-363,Payment,Penghapusan Denda,Installment/Payment Gateway,Penghapusan Denda,Bagaimana cara menghilangkan denda keterlambatan?,Kula informasikan bahwa denda keterlambatan mu...,Kula informasikan bahwa denda keterlambatan mu...
249,2022-12-01,AFI,1134-1140-1163-365,Pengkinian Data,"Tanda tangan elektronik, terkendala OTP dikiri...",Tanda Tangan Elektronik,kode OTP tanda tangan elektronik dikirim ke no...,Bagaimana jika kode OTP tanda tangan elektroni...,Jika Kakak ingin melakukan perubahan nomor tel...,
4,2022-12-01,AFI,738-739-758-121,Keamanan Akun,Akun app - Blokir permanen,Keamanan Akun,Blokir limit permanen,Bagaimana cara mengajukan blokir akun permanen...,Perlu diketahui bahwa setelah Anda melakukan p...,Kula bantu informasikan bahwa setelah Kakak me...
896,NaT,AFI,872-1809-1810-1211\t,Payment,Bayar tagihan Akulaku Paylater pakai rekening ...,Akulaku Paylater,Bayar tagihan Akulaku Paylater pakai rekening ...,min kalo mau bayar tagihan Akulaku Paylater na...,,"Tentu, pembayaran tagihan bisa dilakukan mengg..."


## Duplicate Checking

In [29]:
sheet_pengerjaan = client.open('‌Platform LLM Test').worksheet('Pelaporan Masalah')
data = sheet_pengerjaan.get_all_values()

temp = pd.DataFrame(data[3:], columns=data[2])

# clean the columns
temp.columns = (
    temp.columns
    .map(remove_mandarin)
    .str.replace('\n', '')
    .str.replace(' ','_')
    .str.replace(r'[^A-Za-z0-9_()-]', '', regex=True)
    .str.replace('-','_')
    .str.lower()
    .str.strip('_')
)

temp_imputate = temp[temp['code'].duplicated(keep=False)].sort_values('tanggal_pelaporan', ascending=True)

temp_imputate['tanggal_pelaporan'] = temp_imputate['tanggal_pelaporan'].replace('', 'Null')
temp_imputate[temp_imputate['tanggal_pelaporan'] == 'Null']

,tanggal_pelaporan,pelapor,skenario_bisnis,code,ringkasan_konteks,kategori_2,kategori_3,background_detail,input_pengguna,jawaban_aktual_model,...,tingkat_keparahan,kategori_masalah,kategori_akar_masalah,deskripsi_detail_masalah,prioritas_dev,apakah_stabil_direproduksi,saran_perbaikan,tim_penanggung_jawab,status_penanganan,pic


## New Mod

In [123]:
sheet = client.open('‌Platform LLM Test').worksheet('Pelaporan Masalah')
data = sheet.get_all_values()

df_mod = pd.DataFrame(data[3:], columns=data[2])

# hapus tulisan mandarin
def remove_chinese(text):
    if isinstance(text, str):
        text = re.sub(r'[\u4e00-\u9fff]+', '', text)
        return text.lower().strip()
    return text

# clean nama kolom
df_mod.columns = (
    df_mod.columns
    .str.replace(r'[\u4e00-\u9fff]+', '', regex=True)
    .str.lower()
    .str.strip()
)

# clean isi dataframe
df_mod = (
    df_mod
    .applymap(remove_chinese)
)

# drop duplicate
df_mod = df_mod.drop_duplicates(subset=['code'], keep='first')

df_mod

C:\Users\ozy.prazuganda\AppData\Local\Temp\ipykernel_948\1392918999.py:24: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  .applymap(remove_chinese)


,tanggal pelaporan,pelapor,skenario bisnis,code,ringkasan konteks,kategori_2,kategori_3,background_detail,input pengguna,jawaban aktual model,...,tingkat keparahan,kategori masalah,kategori akar masalah,deskripsi detail masalah,prioritas dev,apakah stabil direproduksi,saran perbaikan,tim penanggung jawab,status penanganan,pic
0,04/30/2026,aulia,akun,1662-1663-1664-996,ingin merubah data diri sesuai dengan data yan...,perubahan data diri,akun,perubahan data diri,1. bagaimana cara merubah data diri yang ada d...,"kak, untuk perubahan data diri seperti nomor k...",...,,,,,,,,,,
1,04/30/2026,aulia,angsuran dan pinjaman,289-1527-1528-922,tolong bantu acc pengajuan pinjaman saya,bantu acc pengajuan pinjaman,pinjaman,bantu acc pengajuan pinjaman,1.) kak saya lagi butuh dana urgent banget. bi...,"kak, maaf ya atas ketidaknyamanannya perihal k...",...,,,,,,,,,,
2,04/30/2026,reza,angsuran dan pinjaman,289-1562-1563-943,kenapa limit pinjaman saya di kurangi,limit pinjaman saya di kurangi,pinjaman,limit pinjaman di kurangi,1. kenapa ya limit saya malah kurang? \n2. lim...,kula paham kekhawatiran kakak. penurunan limit...,...,,,,,,,,,,
3,04/30/2026,reza,payment,289-1667-1668-998,tidak bisa bayar lewat bank bni,keluhkan tidak bisa bayar tagihan lewat bank bni,installment/payment gateway,tidak bisa bayar lewat bank bni,1. kok saya gak bisa bayar lewat bank bni ya s...,kula paham keluhan kakak yang terus mengalami ...,...,,,,,,,,,,
4,04/30/2026,reza,angsuran dan pinjaman,289-1770-1771-1188,kenapa tidak bisa bayar tagihan pinjaman lewat...,tidak bisa bayar pinjaman lewat bca,pinjaman,tidak bisa bayar tagihan pinjaman lewat bca,1. kenapa ini gak bisa bayar tagihan pinjaman ...,"sebelumnya, mohon maaf atas ketidaknyamanannya...",...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
281,,,pengembalian barang,1063-1081-1133-733,bagaimana cara proses retur barang?,cara proses retur barang merchant online,merchant online,cara proses retur barang,,,...,,,,,,,,,,
282,,,angsuran dan pinjaman,289-413-570-734,apa perbedaan bpjs ketenagakerjaan dengan bpjs...,beda bpjs ketenagakerjaan dengan bpjs kesehata...,pinjaman,beda bpjs ketenagakerjaan dengan bpjs kesehata...,,,...,,,,,,,,,,
283,,,angsuran dan pinjaman,289-417-574-739,kenapa limit tidak bisa digunakan untuk pinjam...,"kenapa tidak bisa pinjaman, sedangkan limit te...",pinjaman,"tidak bisa pinjaman, sedangkan limit tersedia",,,...,,,,,,,,,,
284,,,angsuran dan pinjaman,289-391-548-637,saat ajukan pinjaman akun terblokir permanen,akun terblokir permanen dan ingin mengajukan p...,pinjaman,saat ajukan pinjaman akun terblokir permanen,,,...,,,,,,,,,,


In [118]:
# function on separating the question

def separate(text):
    if isinstance(text, list):
        return text
    
    if isinstance(text, str):
        return[i.strip() for i in text.split('\n') if i.strip()]
    
    return text

In [124]:
# Explode
df_mod['input pengguna'] = df_mod['input pengguna'].apply(separate)

df_explode = df_mod.explode('input pengguna').reset_index(drop=True)
df_explode

,tanggal pelaporan,pelapor,skenario bisnis,code,ringkasan konteks,kategori_2,kategori_3,background_detail,input pengguna,jawaban aktual model,...,tingkat keparahan,kategori masalah,kategori akar masalah,deskripsi detail masalah,prioritas dev,apakah stabil direproduksi,saran perbaikan,tim penanggung jawab,status penanganan,pic
0,04/30/2026,aulia,akun,1662-1663-1664-996,ingin merubah data diri sesuai dengan data yan...,perubahan data diri,akun,perubahan data diri,1. bagaimana cara merubah data diri yang ada d...,"kak, untuk perubahan data diri seperti nomor k...",...,,,,,,,,,,
1,04/30/2026,aulia,akun,1662-1663-1664-996,ingin merubah data diri sesuai dengan data yan...,perubahan data diri,akun,perubahan data diri,2.. apakah bisa saya merubah nik ktp saya di a...,"kak, untuk perubahan data diri seperti nomor k...",...,,,,,,,,,,
2,04/30/2026,aulia,akun,1662-1663-1664-996,ingin merubah data diri sesuai dengan data yan...,perubahan data diri,akun,perubahan data diri,3. alamat rumah saya saat ini tidak sesuai de...,"kak, untuk perubahan data diri seperti nomor k...",...,,,,,,,,,,
3,04/30/2026,aulia,angsuran dan pinjaman,289-1527-1528-922,tolong bantu acc pengajuan pinjaman saya,bantu acc pengajuan pinjaman,pinjaman,bantu acc pengajuan pinjaman,1.) kak saya lagi butuh dana urgent banget. bi...,"kak, maaf ya atas ketidaknyamanannya perihal k...",...,,,,,,,,,,
4,04/30/2026,aulia,angsuran dan pinjaman,289-1527-1528-922,tolong bantu acc pengajuan pinjaman saya,bantu acc pengajuan pinjaman,pinjaman,bantu acc pengajuan pinjaman,"2.) saya sudah ajukan pengajuan pinjaman, kok ...","kak, maaf ya atas ketidaknyamanannya perihal k...",...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
746,,,pengembalian barang,1063-1081-1133-733,bagaimana cara proses retur barang?,cara proses retur barang merchant online,merchant online,cara proses retur barang,NaN,,...,,,,,,,,,,
747,,,angsuran dan pinjaman,289-413-570-734,apa perbedaan bpjs ketenagakerjaan dengan bpjs...,beda bpjs ketenagakerjaan dengan bpjs kesehata...,pinjaman,beda bpjs ketenagakerjaan dengan bpjs kesehata...,NaN,,...,,,,,,,,,,
748,,,angsuran dan pinjaman,289-417-574-739,kenapa limit tidak bisa digunakan untuk pinjam...,"kenapa tidak bisa pinjaman, sedangkan limit te...",pinjaman,"tidak bisa pinjaman, sedangkan limit tersedia",NaN,,...,,,,,,,,,,
749,,,angsuran dan pinjaman,289-391-548-637,saat ajukan pinjaman akun terblokir permanen,akun terblokir permanen dan ingin mengajukan p...,pinjaman,saat ajukan pinjaman akun terblokir permanen,NaN,,...,,,,,,,,,,


In [128]:
rows = []

for _, row in df_mod.iterrows():

    exploded_text = separate(row['input pengguna'])

    # row pertama
    first_row = row.to_dict()
    first_row['input pengguna'] = exploded_text[0] if len(exploded_text) > 0 else np.nan

    rows.append(first_row)

    # row tambahan kosong
    for text in exploded_text[1:]:

        empty_row = {col: np.nan for col in df_mod.columns}
        empty_row['input pengguna'] = text

        rows.append(empty_row)

df_final = pd.DataFrame(rows).reset_index(drop=True)

df_final['pelapor'] = df_final['pelapor'].ffill()
df_final['tanggal pelaporan'] = df_final['tanggal pelaporan'].ffill()
df_final

,tanggal pelaporan,pelapor,skenario bisnis,code,ringkasan konteks,kategori_2,kategori_3,background_detail,input pengguna,jawaban aktual model,...,tingkat keparahan,kategori masalah,kategori akar masalah,deskripsi detail masalah,prioritas dev,apakah stabil direproduksi,saran perbaikan,tim penanggung jawab,status penanganan,pic
0,04/30/2026,aulia,akun,1662-1663-1664-996,ingin merubah data diri sesuai dengan data yan...,perubahan data diri,akun,perubahan data diri,1. bagaimana cara merubah data diri yang ada d...,"kak, untuk perubahan data diri seperti nomor k...",...,,,,,,,,,,
1,04/30/2026,aulia,NaN,NaN,NaN,NaN,NaN,NaN,2.. apakah bisa saya merubah nik ktp saya di a...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,04/30/2026,aulia,NaN,NaN,NaN,NaN,NaN,NaN,3. alamat rumah saya saat ini tidak sesuai de...,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,04/30/2026,aulia,angsuran dan pinjaman,289-1527-1528-922,tolong bantu acc pengajuan pinjaman saya,bantu acc pengajuan pinjaman,pinjaman,bantu acc pengajuan pinjaman,1.) kak saya lagi butuh dana urgent banget. bi...,"kak, maaf ya atas ketidaknyamanannya perihal k...",...,,,,,,,,,,
4,04/30/2026,aulia,NaN,NaN,NaN,NaN,NaN,NaN,"2.) saya sudah ajukan pengajuan pinjaman, kok ...",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
746,,,pengembalian barang,1063-1081-1133-733,bagaimana cara proses retur barang?,cara proses retur barang merchant online,merchant online,cara proses retur barang,NaN,,...,,,,,,,,,,
747,,,angsuran dan pinjaman,289-413-570-734,apa perbedaan bpjs ketenagakerjaan dengan bpjs...,beda bpjs ketenagakerjaan dengan bpjs kesehata...,pinjaman,beda bpjs ketenagakerjaan dengan bpjs kesehata...,NaN,,...,,,,,,,,,,
748,,,angsuran dan pinjaman,289-417-574-739,kenapa limit tidak bisa digunakan untuk pinjam...,"kenapa tidak bisa pinjaman, sedangkan limit te...",pinjaman,"tidak bisa pinjaman, sedangkan limit tersedia",NaN,,...,,,,,,,,,,
749,,,angsuran dan pinjaman,289-391-548-637,saat ajukan pinjaman akun terblokir permanen,akun terblokir permanen dan ingin mengajukan p...,pinjaman,saat ajukan pinjaman akun terblokir permanen,NaN,,...,,,,,,,,,,


### Awl

In [ ]:
awl = df_final[df_final['pelapor'] == 'aulia']
awl.to_csv('temp/aul_fill.csv')

In [132]:
awl_unfill = df_final[(df_final['pelapor'] != 'aulia') | (df_final['pelapor'] == '')]
awl_unfill.to_csv('temp/aul_unfill.csv')

### Reza

In [133]:
reza = df_final[df_final['pelapor'] == 'reza']
reza.to_csv('temp/reza_fill.csv')

In [134]:
reza_unfill = df_final[(df_final['pelapor'] != 'reza') | (df_final['pelapor'] == '')]
reza_unfill.to_csv('temp/reza_unfill.csv')